# Visão da carteira por MOB -- macro e micro

Complementa o notebook principal (`ciclo_credito_v8`) com uma visão focada em
**profundidade de janela e MOB**, na linha do que discutimos: uma visão
**macro** (o retrato da base inteira, uma vez) e uma visão **micro**
interativa (uma foto de cada vez, com slider).

**Pré-requisito:** este notebook assume que `df_painel_confiavel` e
`df_originacao` já existem na sessão (produzidos pelo notebook principal,
via `construir_tabelas_a_partir_de_dados_brutos_spark` + o filtro de
óbito/atraso extremo). Roda depois dele, na mesma sessão do cluster, ou
copiando essas duas tabelas antes.

**Instalar o Plotly, se ainda não estiver disponível no cluster:**
```
%pip install plotly
```
(rodar em uma célula antes desta, se necessário -- o resto do notebook
principal usa só `matplotlib`; este aqui é o primeiro a usar `plotly`,
pela interatividade do slider na visão micro.)


In [ ]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots


## Visão macro

Três peças, cada uma olhando a base inteira de uma vez:

1. **Mapa de calor `ref_month` × MOB** -- quanto da base está disponível em
   cada combinação de foto/idade (cor = % da foto, não contagem bruta, pra
   não confundir "carteira cresceu" com "cobertura de MOB mudou"), com duas
   linhas de fronteira: o MOB máximo já observado, e o MOB máximo com
   cobertura "confiável" (acima de um limiar de % da foto).
2. **Escore médio na origem, por safra** -- sempre travado no MOB=1, nunca
   lido no MOB corrente (mesma regra que já aplicamos no Oaxaca: o escore
   atualiza a cada foto, então ler solto misturaria "perfil de quem entrou"
   com deterioração que aconteceu depois).
3. **MOB médio da carteira viva, por foto** -- mostra se a carteira está
   "envelhecendo" (originação desacelerou) ou "rejuvenescendo" (crescimento
   rápido diluindo a idade média).


In [ ]:
def preparar_mapa_calor_mob_spark(df_painel, mob_maximo: int = 24) -> pd.DataFrame:
    """
    Agrega contagem de contratos por (ref_month, months_on_book) -- traz
    tanto a contagem BRUTA quanto o % DENTRO DE CADA FOTO (normalizado por
    ref_month). O % e o que decide a cor do mapa depois -- normalizar
    evita que o crescimento da carteira ao longo do tempo (mais contrato
    no total) se confunda com "mais cobertura de MOB", que e a pergunta
    que esse grafico existe pra responder.
    """
    contagem = (
        df_painel.filter(F.col("months_on_book") <= mob_maximo)
        .groupBy("ref_month", "months_on_book").agg(F.count(F.lit(1)).alias("n_contratos"))
    )
    total_por_foto = df_painel.groupBy("ref_month").agg(F.count(F.lit(1)).alias("n_total_foto"))
    resultado = contagem.join(total_por_foto, on="ref_month").withColumn(
        "pct_da_foto", F.col("n_contratos") / F.col("n_total_foto")
    )
    return resultado.orderBy("ref_month", "months_on_book").toPandas()


def plot_mapa_calor_mob(tabela: pd.DataFrame, limiar_pct_confiavel: float = 0.01):
    """
    Mapa de calor ref_month x MOB (cor = % da foto, nao contagem bruta) +
    duas linhas de fronteira: o MOB maximo ja observado em cada foto
    (vermelho solido), e o MOB maximo com cobertura "confiavel" -- pelo
    menos `limiar_pct_confiavel` da foto (vermelho tracejado). A diferenca
    entre as duas linhas mostra onde existe dado, mas pouco -- regiao pra
    usar com cautela, nao pra descartar de vez.
    """
    pivot_pct = tabela.pivot(index="ref_month", columns="months_on_book", values="pct_da_foto").fillna(0)
    pivot_n = tabela.pivot(index="ref_month", columns="months_on_book", values="n_contratos").fillna(0)

    fronteira_max = tabela.groupby("ref_month")["months_on_book"].max().reset_index()
    fronteira_confiavel = (
        tabela[tabela["pct_da_foto"] >= limiar_pct_confiavel]
        .groupby("ref_month")["months_on_book"].max().reset_index()
    )

    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        z=pivot_pct.values, x=pivot_pct.columns, y=pivot_pct.index.astype(str),
        customdata=pivot_n.values,
        hovertemplate="Foto: %{y}<br>MOB: %{x}<br>%% da foto: %{z:.2%}<br>Contratos: %{customdata:,.0f}<extra></extra>",
        colorscale="Blues", colorbar=dict(title="% da foto", tickformat=".0%"),
    ))
    fig.add_trace(go.Scatter(
        x=fronteira_max["months_on_book"], y=fronteira_max["ref_month"].astype(str),
        mode="lines", line=dict(color="#F2554A", width=2), name="MOB máximo já observado",
    ))
    fig.add_trace(go.Scatter(
        x=fronteira_confiavel["months_on_book"], y=fronteira_confiavel["ref_month"].astype(str),
        mode="lines", line=dict(color="#F2554A", width=2, dash="dash"),
        name=f"MOB confiável (>={limiar_pct_confiavel:.1%} da foto)",
    ))
    fig.update_layout(
        title="Cobertura de MOB por foto mensal (% dos contratos daquela foto)",
        xaxis_title="Months on Book", yaxis_title="Mês de referência (foto)",
        height=700,
    )
    return fig


In [ ]:
def preparar_escore_medio_por_safra_spark(df_painel, coluna_escore: str) -> pd.DataFrame:
    """
    Escore medio na ORIGEM (MOB=1) de cada safra -- mesma regra do Oaxaca:
    nunca ler o escore no MOB corrente (ele atualiza a cada foto), sempre
    travado no nascimento do contrato, senao "qualidade de quem entrou"
    fica contaminada com deterioracao que aconteceu depois.
    """
    resultado = (
        df_painel.filter(F.col("months_on_book") == 1)
        .groupBy("safra").agg(F.avg(coluna_escore).alias("escore_medio_origem"), F.count(F.lit(1)).alias("n_contratos"))
        .orderBy("safra")
    )
    return resultado.toPandas()


def plot_escore_medio_por_safra(tabela: pd.DataFrame, coluna_escore_label: str = "Escore"):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(tabela["safra"].astype(str), tabela["escore_medio_origem"], marker="o", color="#4C72B0")
    ax.set_xlabel("Safra"); ax.set_ylabel(f"{coluna_escore_label} médio na origem")
    ax.set_title(f"{coluna_escore_label} médio na origem, por safra")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    return fig


In [ ]:
def preparar_mob_medio_por_foto_spark(df_painel) -> pd.DataFrame:
    """MOB medio de todos os contratos vivos em cada foto -- mostra se a carteira esta "envelhecendo" (originacao desacelerou) ou "rejuvenescendo" (crescimento rapido diluindo a idade media)."""
    resultado = df_painel.groupBy("ref_month").agg(F.avg("months_on_book").alias("mob_medio"), F.count(F.lit(1)).alias("n_contratos")).orderBy("ref_month")
    return resultado.toPandas()


def plot_mob_medio_por_foto(tabela: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(tabela["ref_month"].astype(str), tabela["mob_medio"], marker="o", color="#DD8452")
    ax.set_xlabel("Mês de referência (foto)"); ax.set_ylabel("MOB médio da carteira viva")
    ax.set_title("Idade média da carteira, ao longo do tempo")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    return fig


## Visão micro interativa

Uma figura só, com **slider por `ref_month`**, sincronizando 6 painéis (7
traços): distribuição de MOB, composição de estágio, risco (atraso 90+)
por MOB, escore atual vs. na origem, `taxa_efetiva`, `renda` -- todos
recalculados juntos quando você arrasta o slider pra outra foto.

Tudo é preparado **uma vez, pra todas as fotos**, em Spark -- o slider só
troca qual linha de cada tabela já pronta é exibida, nunca refaz
agregação. `taxa_efetiva`/`renda` aqui são lidas soltas no MOB corrente de
propósito -- esta visão é o retrato de uma foto, não uma análise causal
entre períodos (onde travar na origem seria obrigatório, como no Oaxaca).


In [ ]:
def preparar_dados_visao_micro_spark(df_painel, coluna_escore: str, coluna_taxa: str = "taxa_efetiva") -> dict:
    """
    Prepara, para TODAS as fotos de uma vez (nao uma query por mes -- caro
    demais), as 6 tabelas pequenas que alimentam os paineis da visao
    micro interativa. Cada uma ja sai agregada por ref_month (e MOB, onde
    fizer sentido) -- o slider so troca QUAL linha de cada tabela mostrar,
    nunca refaz calculo em Spark.

    Dica: se `df_painel` nao estiver em cache/checkpoint, vale um
    `.persist()` antes de chamar esta funcao -- ela varre a mesma tabela
    varias vezes.
    """
    dist_mob = preparar_mapa_calor_mob_spark(df_painel, mob_maximo=24)

    contagem_estagio = df_painel.groupBy("ref_month", "estagio_fonte").agg(F.count(F.lit(1)).alias("n"))
    total_foto = df_painel.groupBy("ref_month").agg(F.count(F.lit(1)).alias("n_total"))
    comp_estagio = contagem_estagio.join(total_foto, on="ref_month").withColumn("pct", F.col("n") / F.col("n_total")).toPandas()

    risco_mob = (
        df_painel.withColumn("atraso90", (F.col("atraso_bucket") == "90+").cast("double"))
        .groupBy("ref_month", "months_on_book").agg(F.avg("atraso90").alias("taxa_atraso90"))
        .toPandas()
    )

    escore_origem = df_painel.filter(F.col("months_on_book") == 1).select("contract_id", F.col(coluna_escore).alias("escore_origem"))
    com_origem = df_painel.join(escore_origem, on="contract_id", how="left")
    escore_box = (
        com_origem.groupBy("ref_month").agg(
            F.percentile_approx(coluna_escore, 0.25).alias("q1"), F.percentile_approx(coluna_escore, 0.5).alias("median"),
            F.percentile_approx(coluna_escore, 0.75).alias("q3"), F.min(coluna_escore).alias("lowerfence"), F.max(coluna_escore).alias("upperfence"),
            F.percentile_approx("escore_origem", 0.25).alias("q1_origem"), F.percentile_approx("escore_origem", 0.5).alias("median_origem"),
            F.percentile_approx("escore_origem", 0.75).alias("q3_origem"), F.min("escore_origem").alias("lowerfence_origem"), F.max("escore_origem").alias("upperfence_origem"),
        ).toPandas()
    )

    def _box_por_foto(coluna):
        return df_painel.groupBy("ref_month").agg(
            F.percentile_approx(coluna, 0.25).alias("q1"), F.percentile_approx(coluna, 0.5).alias("median"),
            F.percentile_approx(coluna, 0.75).alias("q3"), F.min(coluna).alias("lowerfence"), F.max(coluna).alias("upperfence"),
        ).toPandas()

    taxa_box = _box_por_foto(coluna_taxa)
    renda_box = _box_por_foto("renda")

    return {"dist_mob": dist_mob, "comp_estagio": comp_estagio, "risco_mob": risco_mob,
            "escore_box": escore_box, "taxa_box": taxa_box, "renda_box": renda_box}


def plot_visao_micro_interativa(dados: dict, coluna_escore_label: str = "Escore"):
    """Figura unica com slider por ref_month, sincronizando 6 paineis (7 tracos)."""
    meses = sorted(dados["dist_mob"]["ref_month"].unique())
    meses_str = [pd.Timestamp(m).strftime("%Y-%m") for m in meses]

    def _linha_unica(tabela, m):
        return tabela[tabela["ref_month"] == m].iloc[0]

    def frame_data(m):
        sub_mob = dados["dist_mob"][dados["dist_mob"]["ref_month"] == m].sort_values("months_on_book")
        sub_est = dados["comp_estagio"][dados["comp_estagio"]["ref_month"] == m].sort_values("estagio_fonte")
        sub_risco = dados["risco_mob"][dados["risco_mob"]["ref_month"] == m].sort_values("months_on_book")
        e = _linha_unica(dados["escore_box"], m)
        t = _linha_unica(dados["taxa_box"], m)
        r = _linha_unica(dados["renda_box"], m)

        return [
            go.Bar(x=sub_mob["months_on_book"], y=sub_mob["pct_da_foto"]),
            go.Bar(x=[f"Estágio {int(v)}" for v in sub_est["estagio_fonte"]], y=sub_est["pct"]),
            go.Scatter(x=sub_risco["months_on_book"], y=sub_risco["taxa_atraso90"], mode="lines+markers"),
            go.Box(name="Atual", q1=[e["q1"]], median=[e["median"]], q3=[e["q3"]], lowerfence=[e["lowerfence"]], upperfence=[e["upperfence"]]),
            go.Box(name="Na origem", q1=[e["q1_origem"]], median=[e["median_origem"]], q3=[e["q3_origem"]], lowerfence=[e["lowerfence_origem"]], upperfence=[e["upperfence_origem"]]),
            go.Box(name="taxa_efetiva", q1=[t["q1"]], median=[t["median"]], q3=[t["q3"]], lowerfence=[t["lowerfence"]], upperfence=[t["upperfence"]]),
            go.Box(name="renda", q1=[r["q1"]], median=[r["median"]], q3=[r["q3"]], lowerfence=[r["lowerfence"]], upperfence=[r["upperfence"]]),
        ]

    fig = make_subplots(rows=2, cols=3, subplot_titles=(
        "Distribuição de MOB", "Composição de estágio", "Risco (atraso 90+) por MOB",
        f"{coluna_escore_label}: atual vs. origem", "taxa_efetiva", "renda",
    ))

    t0 = frame_data(meses[0])
    posicoes = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 1), (2, 2), (2, 3)]
    for trace, (r_, c_) in zip(t0, posicoes):
        fig.add_trace(trace, row=r_, col=c_)

    fig.frames = [go.Frame(data=frame_data(m), name=m_str) for m, m_str in zip(meses, meses_str)]
    fig.update_layout(
        height=750, showlegend=False, title="Visão micro -- uma foto por vez",
        sliders=[{
            "steps": [{"args": [[m_str], {"frame": {"duration": 0}, "mode": "immediate"}], "label": m_str, "method": "animate"} for m_str in meses_str],
            "currentvalue": {"prefix": "Foto: "},
        }],
    )
    return fig


## Como usar

Roda as funções acima, depois chama nessa ordem (ajuste os nomes de
coluna de escore/taxa para os reais da sua base):


In [ ]:
# visao macro
tabela_mapa_calor = preparar_mapa_calor_mob_spark(df_painel_confiavel)
fig_mapa_calor = plot_mapa_calor_mob(tabela_mapa_calor)
fig_mapa_calor.show()

escore_por_safra = preparar_escore_medio_por_safra_spark(df_painel_confiavel, "score_dos_score")
fig_escore_safra = plot_escore_medio_por_safra(escore_por_safra, "score_dos_score")

mob_medio = preparar_mob_medio_por_foto_spark(df_painel_confiavel)
fig_mob_medio = plot_mob_medio_por_foto(mob_medio)

# visao micro interativa
dados_micro = preparar_dados_visao_micro_spark(df_painel_confiavel, coluna_escore="score_dos_score", coluna_taxa="taxa_efetiva")
fig_micro = plot_visao_micro_interativa(dados_micro, coluna_escore_label="score_dos_score")
fig_micro.show()


---
**Nota de transparência sobre o que foi testado:** a mecânica do slider/
`frames` do Plotly (a parte tecnicamente nova nessa conversa -- o resto do
notebook principal usa só `matplotlib`) foi validada isoladamente, com
dado fabricado em pandas puro, confirmando que os 7 traços e os `frames`
se geram e sincronizam sem erro. As agregações em Spark aqui seguem
exatamente o mesmo padrão (window functions, `groupBy`, `percentile_approx`)
já usado e validado em dezenas de outras funções do notebook principal --
mas, diferente das versões anteriores do pipeline, este notebook em si
**não foi executado de ponta a ponta contra Spark real** nesta sessão
(o ambiente onde eu testei não tem PySpark disponível). Vale rodar a
primeira célula de cada bloco com atenção, antes de confiar no restante.
